In [1]:
import tensorflow as tf
from tensorflow import keras
from keras import layers, models
import pandas as pd
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import time
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
from keras.callbacks import EarlyStopping,ReduceLROnPlateau
from matplotlib import pyplot as plt

In [2]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"keshrivishal21","key":"4c8bb86460f4dd386e694fd83bc26447"}'}

In [3]:
!mkdir -p ~/.config/kaggle
!mv kaggle.json ~/.config/kaggle/
!chmod 600 ~/.config/kaggle/kaggle.json

In [4]:
!kaggle competitions download -c state-farm-distracted-driver-detection

100% 4.00G/4.00G [02:59<00:00, 24.0MB/s]



In [5]:
!unzip -q state-farm-distracted-driver-detection.zip -d data

In [6]:
df = pd.read_csv("/content/data/driver_imgs_list.csv")
df.head()

,subject,classname,img
0,p002,c0,img_44733.jpg
1,p002,c0,img_72999.jpg
2,p002,c0,img_25094.jpg
3,p002,c0,img_69092.jpg
4,p002,c0,img_92629.jpg


In [7]:
drivers = df['subject'].unique()
print("Total drivers:", len(drivers))

Total drivers: 26


In [8]:
train_drivers, temp_drivers = train_test_split(
    drivers,
    test_size=0.4,
    random_state=42
)

val_drivers, test_drivers = train_test_split(
    temp_drivers,
    test_size=0.5,
    random_state=42
)

print("Train drivers:", len(train_drivers))
print("Val drivers:", len(val_drivers))
print("Test drivers:", len(test_drivers))

Train drivers: 15
Val drivers: 5
Test drivers: 6


In [9]:
train_df = df[df['subject'].isin(train_drivers)]
val_df = df[df['subject'].isin(val_drivers)]
test_df = df[df['subject'].isin(test_drivers)]

print(train_df['subject'].nunique())
print(val_df['subject'].nunique())
print(test_df['subject'].nunique())

15
5
6


In [10]:
base_path = "/content/data/imgs/train"

def add_image_path(df):
    return df.apply(
        lambda row: os.path.join(base_path, row['classname'], row['img']),
        axis=1
    )

train_paths = add_image_path(train_df)
val_paths = add_image_path(val_df)
test_paths = add_image_path(test_df)

train_labels = train_df['classname']
val_labels = val_df['classname']
test_labels = test_df['classname']

In [11]:
le = LabelEncoder()
train_labels = le.fit_transform(train_labels)
val_labels = le.transform(val_labels)
test_labels = le.transform(test_labels)

In [ ]:
def mixup(images, labels, alpha=0.2):
    batch_size = tf.shape(images)[0]

    indices = tf.random.shuffle(tf.range(batch_size))
    shuffled_images = tf.gather(images, indices)
    shuffled_labels = tf.gather(labels, indices)

    lam = tf.random.uniform([], 0, 1)

    images = lam * images + (1 - lam) * shuffled_images
    labels = lam * labels + (1 - lam) * shuffled_labels

    return images, labels

In [12]:
IMG_SIZE = (224,224)
BATCH_SIZE = 16
tf.random.set_seed(42)

def process_image(path, label):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3, try_recover_truncated=True)

    h = tf.shape(image)[0]
    w = tf.shape(image)[1]

    # 🔥 ROI cropping (IMPORTANT)
    top = tf.cast(0.20 * tf.cast(h, tf.float32), tf.int32)
    bottom = tf.cast(0.95 * tf.cast(h, tf.float32), tf.int32)
    left = tf.cast(0.10 * tf.cast(w, tf.float32), tf.int32)
    right = tf.cast(0.90 * tf.cast(w, tf.float32), tf.int32)

    image = image[top:bottom, left:right]

    image = tf.image.resize(image, IMG_SIZE, method='bilinear', antialias=True)
    image = tf.cast(image, tf.float32)

    return image, label


def create_dataset(paths, labels, shuffle=True):
    labels = tf.convert_to_tensor(labels, dtype=tf.int32)
    labels = tf.one_hot(labels, depth=10)

    paths = paths.values

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    ds = ds.map(process_image, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE, drop_remainder=True)
    if shuffle:
        ds = ds.shuffle(buffer_size=1500, reshuffle_each_iteration=True)
        ds = ds.map(lambda x, y: mixup(x, y), num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.prefetch(tf.data.AUTOTUNE)

    # 🔥 Performance optimization
    options = tf.data.Options()
    options.experimental_deterministic = False
    ds = ds.with_options(options)

    return ds


train_ds = create_dataset(train_paths, train_labels, shuffle=True)
val_ds = create_dataset(val_paths, val_labels, shuffle=False)
test_ds = create_dataset(test_paths, test_labels, shuffle=False)

In [13]:
for path in train_paths[:5]:
    print(path, os.path.exists(path))

/content/data/imgs/train/c0/img_48693.jpg True
/content/data/imgs/train/c0/img_44903.jpg True
/content/data/imgs/train/c0/img_58514.jpg True
/content/data/imgs/train/c0/img_62307.jpg True
/content/data/imgs/train/c0/img_83984.jpg True


In [14]:
set(train_drivers) & set(val_drivers)
set(train_drivers) & set(test_drivers)
set(val_drivers) & set(test_drivers)

set()

In [15]:
for images, labels in train_ds.take(1):
    print("Image batch shape:", images.shape)
    print("Label batch shape:", labels.shape)

Image batch shape: (16, 224, 224, 3)
Label batch shape: (16, 10)


In [16]:
print("Train images:", len(train_paths))
print("Val images:", len(val_paths))
print("Test images:", len(test_paths))

Train images: 13440
Val images: 3312
Test images: 5672


**Baseline Experiment**

In [ ]:
def mobilenet_driver_model(input_shape=(224,224,3), num_classes=10):

    inputs = layers.Input(shape=input_shape)

    # Augmentation
    x = data_augmentation(inputs)

    # Preprocessing
    x = tf.keras.applications.mobilenet_v2.preprocess_input(x)

    base_model = tf.keras.applications.MobileNetV2(
        input_shape=input_shape,
        include_top=False,
        weights='imagenet'
    )

    base_model.trainable = False

    x = base_model(x, training=False)

    # 🔥 Improved Head
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)

    x = layers.Dense(256, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Dense(128, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)

    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = models.Model(inputs, outputs)

    return model, base_model

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=8,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.3,
    patience=2
)

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.15),
    layers.RandomBrightness(0.1),
])

In [ ]:
model, base_model = mobilenet_driver_model()
model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential (Sequential)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide (TrueDivide)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract (Subtract)             │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,626,762 (10.02 MB)

 Trainable params: 365,450 (1.39 MB)

 Non-trainable params: 2,261,312 (8.63 MB)

In [ ]:

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
    metrics=['accuracy']
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=[early_stop, reduce_lr]
)

Epoch 1/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 111s 102ms/step - accuracy: 0.3304 - loss: 2.1691 - val_accuracy: 0.3345 - val_loss: 2.3330 - learning_rate: 1.0000e-04
Epoch 2/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 134s 108ms/step - accuracy: 0.5604 - loss: 1.4391 - val_accuracy: 0.3989 - val_loss: 2.1889 - learning_rate: 1.0000e-04
Epoch 3/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 88s 98ms/step - accuracy: 0.6604 - loss: 1.1835 - val_accuracy: 0.4327 - val_loss: 2.0601 - learning_rate: 1.0000e-04
Epoch 4/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 88s 97ms/step - accuracy: 0.7193 - loss: 1.0430 - val_accuracy: 0.4674 - val_loss: 1.9832 - learning_rate: 1.0000e-04
Epoch 5/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 100s 112ms/step - accuracy: 0.7516 - loss: 0.9603 - val_accuracy: 0.4819 - val_loss: 1.8887 - learning_rate: 1.0000e-04
Epoch 6/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 130s 99ms/step - accuracy: 0.7821 - loss: 0.8852 - val_accuracy: 0.4970 - val_loss: 1.9143 - learning_rate: 1.0000e-04
Epoch 7/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 87s 9

In [ ]:
test_loss, test_acc = model.evaluate(test_ds)
print("Driver-wise Test Accuracy:", test_acc)

354/354 ━━━━━━━━━━━━━━━━━━━━ 25s 69ms/step - accuracy: 0.5740 - loss: 1.5196
Driver-wise Test Accuracy: 0.5739759802818298


In [ ]:
base_model.trainable = True

# Freeze most layers, train last 40
for layer in base_model.layers[:-40]:
    layer.trainable = False

In [ ]:
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential (Sequential)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide (TrueDivide)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract (Subtract)             │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,357,664 (12.81 MB)

 Trainable params: 2,046,986 (7.81 MB)

 Non-trainable params: 579,776 (2.21 MB)

 Optimizer params: 730,902 (2.79 MB)

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),  # lower Learning rate
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
    metrics=['accuracy']
)
history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=[early_stop, reduce_lr]
)

Epoch 1/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 111s 108ms/step - accuracy: 0.5910 - loss: 1.4024 - val_accuracy: 0.5187 - val_loss: 1.8303 - learning_rate: 1.0000e-05
Epoch 2/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 140s 105ms/step - accuracy: 0.7359 - loss: 1.0039 - val_accuracy: 0.5248 - val_loss: 1.7229 - learning_rate: 1.0000e-05
Epoch 3/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 89s 98ms/step - accuracy: 0.7958 - loss: 0.8581 - val_accuracy: 0.5305 - val_loss: 1.6150 - learning_rate: 1.0000e-05
Epoch 4/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 87s 97ms/step - accuracy: 0.8364 - loss: 0.7618 - val_accuracy: 0.5362 - val_loss: 1.6074 - learning_rate: 1.0000e-05
Epoch 5/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 87s 98ms/step - accuracy: 0.8664 - loss: 0.6914 - val_accuracy: 0.5685 - val_loss: 1.4994 - learning_rate: 1.0000e-05
Epoch 6/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 87s 97ms/step - accuracy: 0.8833 - loss: 0.6555 - val_accuracy: 0.5876 - val_loss: 1.3972 - learning_rate: 1.0000e-05
Epoch 7/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 149s 105

In [ ]:
test_loss, test_acc = model.evaluate(test_ds)
print("Driver-wise Test Accuracy:", test_acc)

473/473 ━━━━━━━━━━━━━━━━━━━━ 15s 32ms/step - accuracy: 0.7664 - loss: 1.0991
Driver-wise Test Accuracy: 0.7663963437080383


In [ ]:
base_model.trainable = True

# Freeze most layers, train last 80
for layer in base_model.layers[:-80]:
    layer.trainable = False

In [ ]:
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential (Sequential)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide (TrueDivide)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract (Subtract)             │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,720,736 (25.64 MB)

 Trainable params: 2,046,986 (7.81 MB)

 Non-trainable params: 579,776 (2.21 MB)

 Optimizer params: 4,093,974 (15.62 MB)

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),  # lower LR
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
    metrics=['accuracy']
)
history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=[early_stop, reduce_lr]
)

Epoch 1/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 103s 102ms/step - accuracy: 0.9297 - loss: 0.5494 - val_accuracy: 0.6325 - val_loss: 1.3118 - learning_rate: 1.0000e-05
Epoch 2/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 95s 105ms/step - accuracy: 0.9395 - loss: 0.5276 - val_accuracy: 0.6165 - val_loss: 1.3772 - learning_rate: 1.0000e-05
Epoch 3/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 94s 105ms/step - accuracy: 0.9432 - loss: 0.5113 - val_accuracy: 0.6144 - val_loss: 1.3664 - learning_rate: 3.0000e-06
Epoch 4/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 87s 97ms/step - accuracy: 0.9445 - loss: 0.5108 - val_accuracy: 0.6196 - val_loss: 1.3536 - learning_rate: 3.0000e-06
Epoch 5/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 94s 104ms/step - accuracy: 0.9470 - loss: 0.5043 - val_accuracy: 0.6223 - val_loss: 1.3447 - learning_rate: 9.0000e-07
Epoch 6/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 93s 105ms/step - accuracy: 0.9495 - loss: 0.4976 - val_accuracy: 0.6205 - val_loss: 1.3415 - learning_rate: 9.0000e-07
Epoch 7/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 143s 1

In [ ]:
test_loss, test_acc = model.evaluate(test_ds)
print("Driver-wise Test Accuracy:", test_acc)

354/354 ━━━━━━━━━━━━━━━━━━━━ 25s 70ms/step - accuracy: 0.6428 - loss: 1.2920
Driver-wise Test Accuracy: 0.6428319215774536


In [ ]:
model.save("/content/data/models/mobileNet_baseline_final.keras")

**Experiment 1 : Adding class weights**

In [ ]:
def mobilenet_driver_model(input_shape=(224,224,3), num_classes=10):

    inputs = layers.Input(shape=input_shape)

    # Augmentation
    x = data_augmentation(inputs)

    # Preprocessing
    x = tf.keras.applications.mobilenet_v2.preprocess_input(x)

    base_model = tf.keras.applications.MobileNetV2(
        input_shape=input_shape,
        include_top=False,
        weights='imagenet'
    )

    base_model.trainable = False

    x = base_model(x, training=False)

    # 🔥 Improved Head
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)

    x = layers.Dense(256, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Dense(128, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)

    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = models.Model(inputs, outputs)

    return model, base_model

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=8,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.3,
    patience=2
)

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.15),
    layers.RandomBrightness(0.1),
])

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels),
    y=train_labels
)

class_weights = dict(enumerate(class_weights))
print(class_weights)

{0: np.float64(0.9155313351498637), 1: np.float64(1.012810851544838), 2: np.float64(0.9696969696969697), 3: np.float64(0.9689978370583994), 4: np.float64(0.9926144756277696), 5: np.float64(0.9767441860465116), 6: np.float64(0.9689978370583994), 7: np.float64(1.1144278606965174), 8: np.float64(1.0971428571428572), 9: np.float64(1.0151057401812689)}


In [ ]:
model, base_model = mobilenet_driver_model()
model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential_1 (Sequential)       │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide_1 (TrueDivide)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract_1 (Subtract)           │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,626,762 (10.02 MB)

 Trainable params: 365,450 (1.39 MB)

 Non-trainable params: 2,261,312 (8.63 MB)

In [ ]:

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
    metrics=['accuracy']
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    class_weight=class_weights,
    epochs=20,
    callbacks=[early_stop, reduce_lr]
)

Epoch 1/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 96s 100ms/step - accuracy: 0.3336 - loss: 2.1516 - val_accuracy: 0.3659 - val_loss: 2.1448 - learning_rate: 1.0000e-04
Epoch 2/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 134s 88ms/step - accuracy: 0.5626 - loss: 1.4335 - val_accuracy: 0.4384 - val_loss: 1.9351 - learning_rate: 1.0000e-04
Epoch 3/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 87s 98ms/step - accuracy: 0.6614 - loss: 1.1897 - val_accuracy: 0.4689 - val_loss: 1.7887 - learning_rate: 1.0000e-04
Epoch 4/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 87s 98ms/step - accuracy: 0.7177 - loss: 1.0567 - val_accuracy: 0.4894 - val_loss: 1.7420 - learning_rate: 1.0000e-04
Epoch 5/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 81s 90ms/step - accuracy: 0.7533 - loss: 0.9602 - val_accuracy: 0.5112 - val_loss: 1.7342 - learning_rate: 1.0000e-04
Epoch 6/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 86s 93ms/step - accuracy: 0.7800 - loss: 0.8890 - val_accuracy: 0.5139 - val_loss: 1.6378 - learning_rate: 1.0000e-04
Epoch 7/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 139s 90ms/

In [ ]:
test_loss, test_acc = model.evaluate(test_ds)
print("Driver-wise Test Accuracy:", test_acc)

354/354 ━━━━━━━━━━━━━━━━━━━━ 26s 72ms/step - accuracy: 0.5293 - loss: 1.6660
Driver-wise Test Accuracy: 0.5293079018592834


In [ ]:
base_model.trainable = True

# Freeze most layers, train last 40
for layer in base_model.layers[:-40]:
    layer.trainable = False

In [ ]:
model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential_1 (Sequential)       │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide_1 (TrueDivide)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract_1 (Subtract)           │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,357,664 (12.81 MB)

 Trainable params: 2,046,986 (7.81 MB)

 Non-trainable params: 579,776 (2.21 MB)

 Optimizer params: 730,902 (2.79 MB)

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),  # lower LR
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
    metrics=['accuracy']
)
history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    class_weight=class_weights,
    epochs=20,
    callbacks=[early_stop, reduce_lr]
)

Epoch 1/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 103s 101ms/step - accuracy: 0.5829 - loss: 1.4220 - val_accuracy: 0.5643 - val_loss: 1.5404 - learning_rate: 1.0000e-05
Epoch 2/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 141s 100ms/step - accuracy: 0.7176 - loss: 1.0524 - val_accuracy: 0.5839 - val_loss: 1.3879 - learning_rate: 1.0000e-05
Epoch 3/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 148s 107ms/step - accuracy: 0.7886 - loss: 0.8731 - val_accuracy: 0.5842 - val_loss: 1.3753 - learning_rate: 1.0000e-05
Epoch 4/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 88s 98ms/step - accuracy: 0.8257 - loss: 0.7835 - val_accuracy: 0.6093 - val_loss: 1.2967 - learning_rate: 1.0000e-05
Epoch 5/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 94s 106ms/step - accuracy: 0.8585 - loss: 0.7110 - val_accuracy: 0.6178 - val_loss: 1.2719 - learning_rate: 1.0000e-05
Epoch 6/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 88s 98ms/step - accuracy: 0.8804 - loss: 0.6616 - val_accuracy: 0.6292 - val_loss: 1.2779 - learning_rate: 1.0000e-05
Epoch 7/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 95s 1

In [ ]:
test_loss, test_acc = model.evaluate(test_ds)
print("Driver-wise Test Accuracy:", test_acc)

354/354 ━━━━━━━━━━━━━━━━━━━━ 25s 69ms/step - accuracy: 0.6810 - loss: 1.1780
Driver-wise Test Accuracy: 0.6809675097465515


In [ ]:
base_model.trainable = True

# Freeze most layers, train last 80
for layer in base_model.layers[:-80]:
    layer.trainable = False

In [ ]:
model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential_1 (Sequential)       │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide_1 (TrueDivide)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract_1 (Subtract)           │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,720,736 (25.64 MB)

 Trainable params: 2,046,986 (7.81 MB)

 Non-trainable params: 579,776 (2.21 MB)

 Optimizer params: 4,093,974 (15.62 MB)

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),  # lower LR
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
    metrics=['accuracy']
)
history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    class_weight=class_weights,
    epochs=20,
    callbacks=[early_stop, reduce_lr]
)

Epoch 1/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 108s 109ms/step - accuracy: 0.9656 - loss: 0.4533 - val_accuracy: 0.6984 - val_loss: 1.1199 - learning_rate: 1.0000e-05
Epoch 2/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 89s 98ms/step - accuracy: 0.9694 - loss: 0.4438 - val_accuracy: 0.7020 - val_loss: 1.0913 - learning_rate: 1.0000e-05
Epoch 3/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 87s 97ms/step - accuracy: 0.9729 - loss: 0.4377 - val_accuracy: 0.7123 - val_loss: 1.0970 - learning_rate: 1.0000e-05
Epoch 4/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 93s 105ms/step - accuracy: 0.9699 - loss: 0.4415 - val_accuracy: 0.7077 - val_loss: 1.0985 - learning_rate: 1.0000e-05
Epoch 5/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 87s 96ms/step - accuracy: 0.9722 - loss: 0.4340 - val_accuracy: 0.7111 - val_loss: 1.1036 - learning_rate: 3.0000e-06
Epoch 6/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 86s 95ms/step - accuracy: 0.9725 - loss: 0.4281 - val_accuracy: 0.7114 - val_loss: 1.0927 - learning_rate: 3.0000e-06
Epoch 7/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 94s 104ms

In [ ]:
test_loss, test_acc = model.evaluate(test_ds)
print("Driver-wise Test Accuracy:", test_acc)

354/354 ━━━━━━━━━━━━━━━━━━━━ 25s 69ms/step - accuracy: 0.6910 - loss: 1.1502
Driver-wise Test Accuracy: 0.6910310983657837


In [ ]:
model.save("/content/data/models/mobileNet_Exp1_final.keras")

**Experiment 2 : Fine Tuning LR**

In [17]:
def mobilenet_driver_model(input_shape=(224,224,3), num_classes=10):

    inputs = layers.Input(shape=input_shape)

    # Augmentation
    x = data_augmentation(inputs)

    # Preprocessing
    x = tf.keras.applications.mobilenet_v2.preprocess_input(x)

    base_model = tf.keras.applications.MobileNetV2(
        input_shape=input_shape,
        include_top=False,
        weights='imagenet'
    )

    base_model.trainable = False

    x = base_model(x, training=False)

    # 🔥 Improved Head
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)

    x = layers.Dense(256, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Dense(128, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)

    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = models.Model(inputs, outputs)

    return model, base_model

In [18]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=8,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.3,
    patience=2
)

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.15),
    layers.RandomBrightness(0.1),
])

In [19]:
model, base_model = mobilenet_driver_model()
model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential (Sequential)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide (TrueDivide)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract (Subtract)             │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,626,762 (10.02 MB)

 Trainable params: 365,450 (1.39 MB)

 Non-trainable params: 2,261,312 (8.63 MB)

In [20]:

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
    metrics=['accuracy']
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=[early_stop, reduce_lr]
)

Epoch 1/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 100s 97ms/step - accuracy: 0.3283 - loss: 2.1649 - val_accuracy: 0.3678 - val_loss: 2.0833 - learning_rate: 1.0000e-04
Epoch 2/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 89s 100ms/step - accuracy: 0.5578 - loss: 1.4474 - val_accuracy: 0.4457 - val_loss: 1.8716 - learning_rate: 1.0000e-04
Epoch 3/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 135s 91ms/step - accuracy: 0.6496 - loss: 1.2003 - val_accuracy: 0.4855 - val_loss: 1.8246 - learning_rate: 1.0000e-04
Epoch 4/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 83s 91ms/step - accuracy: 0.7150 - loss: 1.0371 - val_accuracy: 0.4952 - val_loss: 1.7714 - learning_rate: 1.0000e-04
Epoch 5/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 141s 91ms/step - accuracy: 0.7534 - loss: 0.9595 - val_accuracy: 0.5039 - val_loss: 1.7643 - learning_rate: 1.0000e-04
Epoch 6/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 83s 91ms/step - accuracy: 0.7827 - loss: 0.8864 - val_accuracy: 0.5075 - val_loss: 1.6935 - learning_rate: 1.0000e-04
Epoch 7/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 80s 89ms

In [21]:
test_loss, test_acc = model.evaluate(test_ds)
print("Driver-wise Test Accuracy:", test_acc)

354/354 ━━━━━━━━━━━━━━━━━━━━ 25s 69ms/step - accuracy: 0.5742 - loss: 1.5272
Driver-wise Test Accuracy: 0.5741525292396545


In [22]:
base_model.trainable = True

# Freeze most layers, train last 40
for layer in base_model.layers[:-40]:
    layer.trainable = False

In [23]:
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential (Sequential)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide (TrueDivide)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract (Subtract)             │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,357,664 (12.81 MB)

 Trainable params: 2,046,986 (7.81 MB)

 Non-trainable params: 579,776 (2.21 MB)

 Optimizer params: 730,902 (2.79 MB)

In [24]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
    metrics=['accuracy']
)
history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=[early_stop, reduce_lr]
)

Epoch 1/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 108s 106ms/step - accuracy: 0.6153 - loss: 1.3366 - val_accuracy: 0.5495 - val_loss: 1.7320 - learning_rate: 1.0000e-05
Epoch 2/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 133s 98ms/step - accuracy: 0.7634 - loss: 0.9419 - val_accuracy: 0.5752 - val_loss: 1.4954 - learning_rate: 1.0000e-05
Epoch 3/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 88s 99ms/step - accuracy: 0.8211 - loss: 0.7955 - val_accuracy: 0.5921 - val_loss: 1.3861 - learning_rate: 1.0000e-05
Epoch 4/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 94s 105ms/step - accuracy: 0.8585 - loss: 0.7052 - val_accuracy: 0.6039 - val_loss: 1.3468 - learning_rate: 1.0000e-05
Epoch 5/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 96s 106ms/step - accuracy: 0.8863 - loss: 0.6448 - val_accuracy: 0.6175 - val_loss: 1.3115 - learning_rate: 1.0000e-05
Epoch 6/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 88s 98ms/step - accuracy: 0.8980 - loss: 0.6152 - val_accuracy: 0.6325 - val_loss: 1.2615 - learning_rate: 1.0000e-05
Epoch 7/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 148s 10

In [25]:
test_loss, test_acc = model.evaluate(test_ds)
print("Driver-wise Test Accuracy:", test_acc)

354/354 ━━━━━━━━━━━━━━━━━━━━ 25s 71ms/step - accuracy: 0.6580 - loss: 1.2115
Driver-wise Test Accuracy: 0.6580155491828918


In [26]:
base_model.trainable = True

# Freeze most layers, train last 80
for layer in base_model.layers[:-80]:
    layer.trainable = False

In [27]:
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential (Sequential)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide (TrueDivide)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract (Subtract)             │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,720,736 (25.64 MB)

 Trainable params: 2,046,986 (7.81 MB)

 Non-trainable params: 579,776 (2.21 MB)

 Optimizer params: 4,093,974 (15.62 MB)

In [28]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=3e-6),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
    metrics=['accuracy']
)
history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=[early_stop, reduce_lr]
)

Epoch 1/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 102s 101ms/step - accuracy: 0.9491 - loss: 0.4925 - val_accuracy: 0.6772 - val_loss: 1.1520 - learning_rate: 3.0000e-06
Epoch 2/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 88s 97ms/step - accuracy: 0.9519 - loss: 0.4883 - val_accuracy: 0.6809 - val_loss: 1.1438 - learning_rate: 3.0000e-06
Epoch 3/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 94s 105ms/step - accuracy: 0.9531 - loss: 0.4849 - val_accuracy: 0.6796 - val_loss: 1.1422 - learning_rate: 9.0000e-07
Epoch 4/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 95s 105ms/step - accuracy: 0.9563 - loss: 0.4780 - val_accuracy: 0.6827 - val_loss: 1.1408 - learning_rate: 9.0000e-07
Epoch 5/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 87s 97ms/step - accuracy: 0.9544 - loss: 0.4819 - val_accuracy: 0.6821 - val_loss: 1.1427 - learning_rate: 2.7000e-07
Epoch 6/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 88s 99ms/step - accuracy: 0.9557 - loss: 0.4790 - val_accuracy: 0.6812 - val_loss: 1.1409 - learning_rate: 2.7000e-07
Epoch 7/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 87s 97ms

In [29]:
test_loss, test_acc = model.evaluate(test_ds)
print("Driver-wise Test Accuracy:", test_acc)

354/354 ━━━━━━━━━━━━━━━━━━━━ 25s 70ms/step - accuracy: 0.6524 - loss: 1.2358
Driver-wise Test Accuracy: 0.6523658037185669


In [30]:
model.save("/content/data/models/mobileNet_fineTune_final.keras")

**Experiment 3 : Better augmentation**

In [17]:
def mobilenet_driver_model(input_shape=(224,224,3), num_classes=10):

    inputs = layers.Input(shape=input_shape)

    # Augmentation
    x = data_augmentation(inputs)

    # Preprocessing
    x = tf.keras.applications.mobilenet_v2.preprocess_input(x)

    base_model = tf.keras.applications.MobileNetV2(
        input_shape=input_shape,
        include_top=False,
        weights='imagenet'
    )

    base_model.trainable = False

    x = base_model(x, training=False)

    # 🔥 Improved Head
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)

    x = layers.Dense(256, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Dense(128, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)

    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = models.Model(inputs, outputs)

    return model, base_model

In [18]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=8,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.3,
    patience=2
)

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.15),
    layers.RandomBrightness(0.1),
    layers.GaussianNoise(0.02),
])

In [19]:
model, base_model = mobilenet_driver_model()
model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential (Sequential)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide (TrueDivide)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract (Subtract)             │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,626,762 (10.02 MB)

 Trainable params: 365,450 (1.39 MB)

 Non-trainable params: 2,261,312 (8.63 MB)

In [20]:

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
    metrics=['accuracy']
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=[early_stop, reduce_lr]
)

Epoch 1/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 98s 95ms/step - accuracy: 0.3302 - loss: 2.1582 - val_accuracy: 0.3539 - val_loss: 2.0734 - learning_rate: 1.0000e-04
Epoch 2/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 135s 92ms/step - accuracy: 0.5643 - loss: 1.4339 - val_accuracy: 0.4067 - val_loss: 1.9403 - learning_rate: 1.0000e-04
Epoch 3/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 83s 92ms/step - accuracy: 0.6612 - loss: 1.1738 - val_accuracy: 0.4396 - val_loss: 1.8388 - learning_rate: 1.0000e-04
Epoch 4/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 83s 91ms/step - accuracy: 0.7107 - loss: 1.0580 - val_accuracy: 0.4701 - val_loss: 1.7924 - learning_rate: 1.0000e-04
Epoch 5/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 89s 100ms/step - accuracy: 0.7562 - loss: 0.9486 - val_accuracy: 0.4967 - val_loss: 1.7180 - learning_rate: 1.0000e-04
Epoch 6/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 90s 100ms/step - accuracy: 0.7810 - loss: 0.8908 - val_accuracy: 0.5290 - val_loss: 1.6176 - learning_rate: 1.0000e-04
Epoch 7/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 89s 99ms/

In [21]:
test_loss, test_acc = model.evaluate(test_ds)
print("Driver-wise Test Accuracy:", test_acc)

354/354 ━━━━━━━━━━━━━━━━━━━━ 24s 67ms/step - accuracy: 0.5644 - loss: 1.4805
Driver-wise Test Accuracy: 0.5644420981407166


In [22]:
base_model.trainable = True

# Freeze most layers, train last 40
for layer in base_model.layers[:-40]:
    layer.trainable = False

In [23]:
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential (Sequential)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide (TrueDivide)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract (Subtract)             │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,357,664 (12.81 MB)

 Trainable params: 2,046,986 (7.81 MB)

 Non-trainable params: 579,776 (2.21 MB)

 Optimizer params: 730,902 (2.79 MB)

In [24]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
    metrics=['accuracy']
)
history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=[early_stop, reduce_lr]
)

Epoch 1/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 106s 106ms/step - accuracy: 0.6183 - loss: 1.3260 - val_accuracy: 0.5743 - val_loss: 1.6361 - learning_rate: 1.0000e-05
Epoch 2/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 86s 96ms/step - accuracy: 0.7590 - loss: 0.9413 - val_accuracy: 0.5861 - val_loss: 1.4244 - learning_rate: 1.0000e-05
Epoch 3/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 143s 96ms/step - accuracy: 0.8240 - loss: 0.7875 - val_accuracy: 0.5945 - val_loss: 1.3525 - learning_rate: 1.0000e-05
Epoch 4/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 142s 97ms/step - accuracy: 0.8540 - loss: 0.7191 - val_accuracy: 0.6036 - val_loss: 1.2946 - learning_rate: 1.0000e-05
Epoch 5/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 93s 105ms/step - accuracy: 0.8864 - loss: 0.6507 - val_accuracy: 0.6256 - val_loss: 1.2335 - learning_rate: 1.0000e-05
Epoch 6/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 93s 104ms/step - accuracy: 0.9000 - loss: 0.6118 - val_accuracy: 0.6187 - val_loss: 1.2437 - learning_rate: 1.0000e-05
Epoch 7/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 87s 96

In [25]:
test_loss, test_acc = model.evaluate(test_ds)
print("Driver-wise Test Accuracy:", test_acc)

354/354 ━━━━━━━━━━━━━━━━━━━━ 24s 69ms/step - accuracy: 0.6704 - loss: 1.1944
Driver-wise Test Accuracy: 0.6703742742538452


In [26]:
base_model.trainable = True

# Freeze most layers, train last 80
for layer in base_model.layers[:-80]:
    layer.trainable = False

In [27]:
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential (Sequential)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide (TrueDivide)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract (Subtract)             │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,720,736 (25.64 MB)

 Trainable params: 2,046,986 (7.81 MB)

 Non-trainable params: 579,776 (2.21 MB)

 Optimizer params: 4,093,974 (15.62 MB)

In [28]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
    metrics=['accuracy']
)
history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=[early_stop, reduce_lr]
)

Epoch 1/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 100s 99ms/step - accuracy: 0.9673 - loss: 0.4450 - val_accuracy: 0.6806 - val_loss: 1.1414 - learning_rate: 1.0000e-05
Epoch 2/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 141s 97ms/step - accuracy: 0.9673 - loss: 0.4449 - val_accuracy: 0.6884 - val_loss: 1.1475 - learning_rate: 1.0000e-05
Epoch 3/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 140s 97ms/step - accuracy: 0.9724 - loss: 0.4331 - val_accuracy: 0.6914 - val_loss: 1.1248 - learning_rate: 3.0000e-06
Epoch 4/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 86s 96ms/step - accuracy: 0.9713 - loss: 0.4339 - val_accuracy: 0.6854 - val_loss: 1.1378 - learning_rate: 3.0000e-06
Epoch 5/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 93s 105ms/step - accuracy: 0.9735 - loss: 0.4272 - val_accuracy: 0.6890 - val_loss: 1.1326 - learning_rate: 9.0000e-07
Epoch 6/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 138s 99ms/step - accuracy: 0.9741 - loss: 0.4258 - val_accuracy: 0.6884 - val_loss: 1.1358 - learning_rate: 9.0000e-07
Epoch 7/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 89s 98m

In [29]:
test_loss, test_acc = model.evaluate(test_ds)
print("Driver-wise Test Accuracy:", test_acc)

354/354 ━━━━━━━━━━━━━━━━━━━━ 24s 69ms/step - accuracy: 0.6706 - loss: 1.1867
Driver-wise Test Accuracy: 0.6705508232116699


In [30]:
model.save("/content/data/models/mobileNet_Exp3_final.keras")

**Experiment 4 : Added mix up**

In [ ]:
def mobilenet_driver_model(input_shape=(224,224,3), num_classes=10):

    inputs = layers.Input(shape=input_shape)

    # Augmentation
    x = data_augmentation(inputs)

    # Preprocessing
    x = tf.keras.applications.mobilenet_v2.preprocess_input(x)

    base_model = tf.keras.applications.MobileNetV2(
        input_shape=input_shape,
        include_top=False,
        weights='imagenet'
    )

    base_model.trainable = False

    x = base_model(x, training=False)

    # 🔥 Improved Head
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)

    x = layers.Dense(256, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Dense(128, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)

    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = models.Model(inputs, outputs)

    return model, base_model

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=8,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.3,
    patience=2
)

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.15),
    layers.RandomBrightness(0.1),
])

In [ ]:
model, base_model = mobilenet_driver_model()
model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential (Sequential)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide (TrueDivide)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract (Subtract)             │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,626,762 (10.02 MB)

 Trainable params: 365,450 (1.39 MB)

 Non-trainable params: 2,261,312 (8.63 MB)

In [ ]:

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
    metrics=['accuracy']
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=[early_stop, reduce_lr]
)

Epoch 1/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 111s 102ms/step - accuracy: 0.3304 - loss: 2.1691 - val_accuracy: 0.3345 - val_loss: 2.3330 - learning_rate: 1.0000e-04
Epoch 2/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 134s 108ms/step - accuracy: 0.5604 - loss: 1.4391 - val_accuracy: 0.3989 - val_loss: 2.1889 - learning_rate: 1.0000e-04
Epoch 3/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 88s 98ms/step - accuracy: 0.6604 - loss: 1.1835 - val_accuracy: 0.4327 - val_loss: 2.0601 - learning_rate: 1.0000e-04
Epoch 4/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 88s 97ms/step - accuracy: 0.7193 - loss: 1.0430 - val_accuracy: 0.4674 - val_loss: 1.9832 - learning_rate: 1.0000e-04
Epoch 5/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 100s 112ms/step - accuracy: 0.7516 - loss: 0.9603 - val_accuracy: 0.4819 - val_loss: 1.8887 - learning_rate: 1.0000e-04
Epoch 6/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 130s 99ms/step - accuracy: 0.7821 - loss: 0.8852 - val_accuracy: 0.4970 - val_loss: 1.9143 - learning_rate: 1.0000e-04
Epoch 7/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 87s 9

In [ ]:
test_loss, test_acc = model.evaluate(test_ds)
print("Driver-wise Test Accuracy:", test_acc)

354/354 ━━━━━━━━━━━━━━━━━━━━ 25s 69ms/step - accuracy: 0.5740 - loss: 1.5196
Driver-wise Test Accuracy: 0.5739759802818298


In [ ]:
base_model.trainable = True

# Freeze most layers, train last 40
for layer in base_model.layers[:-40]:
    layer.trainable = False

In [ ]:
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential (Sequential)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide (TrueDivide)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract (Subtract)             │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,357,664 (12.81 MB)

 Trainable params: 2,046,986 (7.81 MB)

 Non-trainable params: 579,776 (2.21 MB)

 Optimizer params: 730,902 (2.79 MB)

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),  # lower Learning rate
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
    metrics=['accuracy']
)
history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=[early_stop, reduce_lr]
)

Epoch 1/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 111s 108ms/step - accuracy: 0.5910 - loss: 1.4024 - val_accuracy: 0.5187 - val_loss: 1.8303 - learning_rate: 1.0000e-05
Epoch 2/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 140s 105ms/step - accuracy: 0.7359 - loss: 1.0039 - val_accuracy: 0.5248 - val_loss: 1.7229 - learning_rate: 1.0000e-05
Epoch 3/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 89s 98ms/step - accuracy: 0.7958 - loss: 0.8581 - val_accuracy: 0.5305 - val_loss: 1.6150 - learning_rate: 1.0000e-05
Epoch 4/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 87s 97ms/step - accuracy: 0.8364 - loss: 0.7618 - val_accuracy: 0.5362 - val_loss: 1.6074 - learning_rate: 1.0000e-05
Epoch 5/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 87s 98ms/step - accuracy: 0.8664 - loss: 0.6914 - val_accuracy: 0.5685 - val_loss: 1.4994 - learning_rate: 1.0000e-05
Epoch 6/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 87s 97ms/step - accuracy: 0.8833 - loss: 0.6555 - val_accuracy: 0.5876 - val_loss: 1.3972 - learning_rate: 1.0000e-05
Epoch 7/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 149s 105

In [ ]:
test_loss, test_acc = model.evaluate(test_ds)
print("Driver-wise Test Accuracy:", test_acc)

473/473 ━━━━━━━━━━━━━━━━━━━━ 15s 32ms/step - accuracy: 0.7664 - loss: 1.0991
Driver-wise Test Accuracy: 0.7663963437080383


In [ ]:
base_model.trainable = True

# Freeze most layers, train last 80
for layer in base_model.layers[:-80]:
    layer.trainable = False

In [ ]:
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential (Sequential)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide (TrueDivide)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract (Subtract)             │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,720,736 (25.64 MB)

 Trainable params: 2,046,986 (7.81 MB)

 Non-trainable params: 579,776 (2.21 MB)

 Optimizer params: 4,093,974 (15.62 MB)

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),  # lower LR
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
    metrics=['accuracy']
)
history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=[early_stop, reduce_lr]
)

Epoch 1/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 103s 102ms/step - accuracy: 0.9297 - loss: 0.5494 - val_accuracy: 0.6325 - val_loss: 1.3118 - learning_rate: 1.0000e-05
Epoch 2/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 95s 105ms/step - accuracy: 0.9395 - loss: 0.5276 - val_accuracy: 0.6165 - val_loss: 1.3772 - learning_rate: 1.0000e-05
Epoch 3/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 94s 105ms/step - accuracy: 0.9432 - loss: 0.5113 - val_accuracy: 0.6144 - val_loss: 1.3664 - learning_rate: 3.0000e-06
Epoch 4/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 87s 97ms/step - accuracy: 0.9445 - loss: 0.5108 - val_accuracy: 0.6196 - val_loss: 1.3536 - learning_rate: 3.0000e-06
Epoch 5/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 94s 104ms/step - accuracy: 0.9470 - loss: 0.5043 - val_accuracy: 0.6223 - val_loss: 1.3447 - learning_rate: 9.0000e-07
Epoch 6/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 93s 105ms/step - accuracy: 0.9495 - loss: 0.4976 - val_accuracy: 0.6205 - val_loss: 1.3415 - learning_rate: 9.0000e-07
Epoch 7/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 143s 1

In [ ]:
test_loss, test_acc = model.evaluate(test_ds)
print("Driver-wise Test Accuracy:", test_acc)

354/354 ━━━━━━━━━━━━━━━━━━━━ 25s 70ms/step - accuracy: 0.6428 - loss: 1.2920
Driver-wise Test Accuracy: 0.6428319215774536


In [ ]:
model.save("/content/data/models/mobileNet_Exp4_final.keras")